# 📈 Bitcoin (BTC) Price Prediction using Time-Series Forecasting

## 1. Business Understanding

### 1.1 Context

Bitcoin is a digital asset known for its highly volatile price movements. Within a short period, its price can rise or fall quite drastically. This differs from traditional assets such as gold, which tend to be more stable.

Bitcoin price changes are influenced by many factors, including market sentiment, blockchain technology developments, government regulations across countries, global economic conditions, and speculative investor behavior. Due to this volatility, Bitcoin becomes an interesting subject to study, particularly in the context of price prediction using historical data.

Bitcoin price prediction provides important benefits, such as helping traders and investors make decisions, manage portfolio risk, and understand medium- to long-term trends.

---

### 1.2 Problem Statement

Stakeholders want to determine whether historical Bitcoin price data can be used to predict future Bitcoin prices, particularly the monthly closing price. The dataset used consists of daily Bitcoin price data from 2014 to 2025.

Several key questions addressed in this project include:

* Can past Bitcoin price data effectively predict future prices?
* Which prediction model performs better for highly volatile price movements?
* How large are the prediction errors produced by each model?

---

### 1.3 Goals

The objective of this project is to build an end-to-end Bitcoin price prediction pipeline using a time-series forecasting approach. More specifically, the goals are:

* Transform daily Bitcoin price data into monthly data to better observe price movement patterns.
* Predict monthly Bitcoin prices for the next two years (24 months).
* Compare prediction results from several different time-series models.
* Evaluate the accuracy of each model to determine which performs best.

In [31]:
# =========================
# Standard Library
# =========================
import os
import sys
import json
import warnings
from datetime import datetime

# =========================
# Third-Party Libraries
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import joblib

# Time Series & Statistics
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA

# Forecasting
from prophet import Prophet

# Evaluation
from sklearn.metrics import mean_squared_error

# =========================
# Configuration
# =========================
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root added to sys.path: {PROJECT_ROOT}")

Project root added to sys.path: c:\Users\U1\Documents\bitcoin-ts-1


## 2. Data Acquisition and Understanding

### 2.1 Load Data

In [32]:
# Load dataset
try:
    df_raw = pd.read_csv('../datasets/btc_2014_2025.csv')
    df = df_raw.copy()
    # Convert 'date' to datetime
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    print("Original Data Shape (Daily):", df.shape)
    display(df.head())
except FileNotFoundError:
    print("Error: File 'btc_2014_2025.csv' not found. Please upload the dataset.")

Original Data Shape (Daily): (4121, 5)


,open,high,low,close,volume
date,,,,,
2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800
2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200
2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700
2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600
2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100


Using the `btc_2014_2025.csv` dataset containing daily Bitcoin **OHLCV** data sourced from Yahoo Finance. This dataset records Bitcoin’s daily price movements, from opening to closing prices, along with trading activity.

In simple terms, OHLCV data consists of:

* **Open**: Bitcoin price at the start of the trading day
* **High**: The highest price reached within the day
* **Low**: The lowest price reached within the day
* **Close**: Bitcoin price at the end of the trading day
* **Volume**: The number of Bitcoin transactions during the day

The dataset covers a long period from **2014 to 2025**. This time span is sufficient to observe long-term Bitcoin price behavior, including periods of sharp growth, significant decline, as well as phases of high and low volatility.

In this project, daily price data is not used directly for prediction. Instead, the data is transformed into **monthly data** to reduce extreme daily fluctuations. This approach makes Bitcoin price movement patterns easier to observe and analyze.

The primary focus of the analysis is on the **closing price** column. The closing price is chosen because it best represents Bitcoin’s value at the end of each trading period and is commonly used as a reference in market analysis and time-series modeling.

After understanding the dataset structure and content, the next step is exploratory data analysis to examine overall trends, seasonal patterns, and fundamental characteristics of Bitcoin price movements before proceeding to the modeling stage.


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4121 entries, 2014-09-17 to 2025-12-29
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    4121 non-null   float64
 1   high    4121 non-null   float64
 2   low     4121 non-null   float64
 3   close   4121 non-null   float64
 4   volume  4121 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 193.2 KB


The output of `df_raw.info()` is used to obtain a general overview of the dataset structure used in this project.

The dataset consists of **4,115 rows**, representing **daily** Bitcoin price data from **September 17, 2014 to December 22, 2025**. Each row corresponds to one Bitcoin trading day.

This dataset contains **five main columns**, namely:

* **open**: Bitcoin price at the start of the trading day
* **high**: the highest Bitcoin price reached within a day
* **low**: the lowest Bitcoin price within a day
* **close**: Bitcoin price at the end of the trading day
* **volume**: the number of Bitcoin transactions on that day

All columns in the dataset contain **complete (non-null) values**, meaning there is no missing data within the analyzed time period. This is important because complete data reduces the need for data cleaning and minimizes potential bias in the analysis results.


In [34]:
df.duplicated().sum()

np.int64(0)

Duplicate data checking was performed to ensure that no records were entered more than once for the same time period. The result of the check shows a value of **0**, indicating that **no duplicate data was found** in the dataset. Each time period contains only one unique Bitcoin price record.

### 2.2 Preprocessing: Resampling to Monthly

In [35]:
# Resample to Monthly Mean price
df = df['close'].resample('MS').mean().to_frame(name='Price')

print("Resampled Data Shape (Monthly):", df.shape)
df.head()

Resampled Data Shape (Monthly): (136, 1)


,Price
date,
2014-09-01,407.182428
2014-10-01,364.148873
2014-11-01,366.099799
2014-12-01,341.267871
2015-01-01,248.782547


Since ARIMA/SARIMA models can become very slow and noisy when using thousands of daily observations, the data is resampled into **monthly frequency (Monthly Start – MS)**. The average `close` price is calculated for each month.

At this stage, Bitcoin price data that was originally **daily** is transformed into **monthly** data. This process is known as *resampling*, which involves grouping data based on a specific time period.

In this case, the following approach is used:

* **Closing price (close)** as the primary focus
* **Monthly period**, by calculating the **average closing price for each month**

The main purpose of this step is to reduce sharp daily fluctuations. Bitcoin’s day-to-day price movements are often extremely volatile and noisy, making it difficult to observe general price patterns.

By converting the data into monthly frequency:

* Medium- and long-term trends become clearer
* Upward and downward price patterns are easier to observe
* Prediction models can operate more stably and are less affected by extreme daily spikes

The result of this process is a new dataset containing a single price value for each month, labeled as **Price**. Each row now represents the average Bitcoin price within a given month.

From the displayed output, the monthly data begins in **September 2014** and continues sequentially at the start of each month. This dataset will serve as the primary foundation for further analysis and time-series modeling in the next stage.


### 2.3 Exploratory Data Analysis (EDA)

In [36]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df['Price'],
        mode='lines',
        name='BTC Monthly Average Price',
        line=dict(color='orange', width=2)
    )
)

fig.update_layout(
    title='Bitcoin Monthly Price Trend (2014 - 2025)',
    xaxis_title='Year',
    yaxis_title='Price (USD)',
    template='plotly_white',
    hovermode='x unified',
    xaxis=dict(
        showgrid=True,
        rangeslider=dict(visible=True)
    ),
    yaxis=dict(showgrid=True)
)

fig.show()

At this stage, visualization is performed to observe the **monthly Bitcoin price trend** from 2014 to 2025. The displayed chart is a *line chart* showing the average Bitcoin price for each month. The horizontal axis (X) represents time (years), while the vertical axis (Y) indicates the Bitcoin price in USD.

The main objectives of this visualization are to:

* Examine the overall direction of Bitcoin price movement over the long term
* Identify periods of significant price increases and declines
* Understand the level of Bitcoin price volatility over time

From the chart, it can be observed that Bitcoin prices experience sharp fluctuations, with several periods of rapid growth followed by significant declines. This pattern highlights Bitcoin’s high volatility characteristics, making it challenging to predict.

### 2.4 Stationarity Check

In [37]:
def adf_test(series):
    result = adfuller(series.dropna())
    print('ADF Statistic: %f' % result[0])
    print('p-value: %f' % result[1])
    if result[1] > 0.05:
        print("Result: Data is Non-Stationary (Series has a unit root)")
    else:
        print("Result: Data is Stationary")

adf_test(df['Price'])

ADF Statistic: -0.780886
p-value: 0.824672
Result: Data is Non-Stationary (Series has a unit root)


Stationarity testing was conducted using the **Augmented Dickey-Fuller (ADF) Test**. This test aims to determine whether Bitcoin price data is **stationary** or not.

In simple terms, data is considered **stationary** if its movement pattern remains relatively stable over time, both in terms of mean and variance. Conversely, data is **non-stationary** if it exhibits a strong upward or downward trend, causing its statistical properties to change over time.

The ADF Test results on the monthly Bitcoin price data indicate:

* A small negative **ADF Statistic**
* A **p-value of 0.82**, which is much higher than the common threshold of 0.05

Based on these results, the data is classified as **non-stationary**. This means Bitcoin prices exhibit long-term trends and significant pattern changes over time.

This stationarity check is particularly important for **ARIMA** models because:

* ARIMA assumes that the input data is stationary
* Non-stationary data can lead to misleading prediction results
* The model may struggle to capture consistent patterns

Since the Bitcoin price data is not yet stationary, a **differencing** process (calculating the change between consecutive time periods) is required before building the ARIMA model. This step aims to remove trends and stabilize the data. By understanding this characteristic early, the modeling process can be performed using an appropriate approach, resulting in more reliable predictions.

## 3. Modeling

### 3.1 Data Preparation (Train-Test Split)

In [38]:
test_size = 24
train_data = df.iloc[:-test_size]
test_data = df.iloc[-test_size:]

print(f"Train period: {train_data.index.min()} to {train_data.index.max()}")
print(f"Test period: {test_data.index.min()} to {test_data.index.max()}")

Train period: 2014-09-01 00:00:00 to 2023-12-01 00:00:00
Test period: 2024-01-01 00:00:00 to 2025-12-01 00:00:00


At this stage, the dataset is divided into two parts: the **training set** and the **test set**. This split aims to evaluate the model’s ability to predict unseen data. The **last 24 months** (equivalent to **2 years**) are used as the test set, while the remaining data is used as the training set. This approach is commonly applied in *time-series* analysis because it reflects real-world conditions, where models are trained on past data and tested on future periods.

Based on the data split results:

* The **training set** covers the period from **September 2014 to December 2023**
* The **test set** covers the period from **January 2024 to December 2025**

The training data is used to learn historical Bitcoin price patterns, such as long-term trends and general fluctuations. Meanwhile, the test data is used to evaluate how well the model can predict Bitcoin price movements in more recent periods.


### 3.2 Model 1: ARIMA

In [ ]:
# Fit ARIMA Model
# Uses the standard order (1,1,1) for demonstration.
# It is recommended to use pmdarima.auto_arima for optimal order.
arima_model = ARIMA(train_data['Price'], order=(1, 1, 1))
arima_fit = arima_model.fit()
print(arima_fit.summary())

                               SARIMAX Results                                
Dep. Variable:                  Price   No. Observations:                  112
Model:                 ARIMA(1, 1, 1)   Log Likelihood               -1054.465
Date:                Mon, 29 Dec 2025   AIC                           2114.931
Time:                        13:40:25   BIC                           2123.059
Sample:                    09-01-2014   HQIC                          2118.228
                         - 12-01-2023                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2179      0.118      1.848      0.065      -0.013       0.449
ma.L1          0.3738      0.080      4.687      0.000       0.218       0.530
sigma2        1.1e+07   8.01e+05     13.737      0.0

At this stage, an **ARIMA** model is developed as the initial (*baseline*) model to predict monthly Bitcoin prices. A baseline model serves as a reference point so that the performance of more complex models can be evaluated fairly. The configuration used is **ARIMA(1,1,1)**, as it is commonly adopted as a starting point in *time-series* analysis.

Conceptually, ARIMA(1,1,1) operates with three main components:

* **Autoregressive (AR)**: the model captures the relationship between the current month’s price and the price from one month earlier
* **Integrated (I)**: the price data is transformed into **month-to-month differences (differencing)** to remove long-term trends and stabilize the data pattern
* **Moving Average (MA)**: the model incorporates prediction errors from previous periods as part of the forecasting process

This approach is chosen because Bitcoin prices exhibit high volatility and sharp trend changes. By applying *differencing* implicitly through the parameter `d = 1`, the model can focus on learning price changes rather than raw price levels that continuously rise or fall over time.

The model summary results indicate that the **Moving Average (MA)** component has a stronger influence compared to the **Autoregressive (AR)** component. This suggests that Bitcoin price dynamics are more affected by short-term fluctuations and previous error corrections rather than stable historical price patterns.

The **AIC** and **BIC** values are presented as indicators of model quality and complexity. In general, lower values suggest a more efficient model in explaining the data. However, at this stage, these values are not yet compared with other ARIMA configurations since this model is used solely as a baseline.

It is important to note that the **(1,1,1)** configuration may not be optimal. To improve performance, parameter selection is typically conducted through systematic search methods such as *grid search* or **AutoARIMA**. Nevertheless, this simple ARIMA model is sufficient to provide an initial understanding of how time-series models perform in predicting Bitcoin prices.

The ARIMA model is then used to generate predictions on the test dataset, allowing its performance to be compared with other models such as SARIMA and Prophet in the subsequent evaluation stage.


In [40]:
# Forecasting ARIMA
arima_forecast = arima_fit.forecast(steps=len(test_data))
arima_rmse = np.sqrt(mean_squared_error(test_data['Price'], arima_forecast))
print(f"ARIMA RMSE: {arima_rmse:.2f}")

ARIMA RMSE: 44018.15


At this stage, the **ARIMA** model is used to perform **Bitcoin price forecasting** on the test dataset. The model generates predictions for the **next 24 months**, matching the length of the previously defined test set.

After generating the forecasts, model performance is evaluated using the **RMSE (Root Mean Squared Error)** metric. RMSE measures the average magnitude of the difference between predicted prices and actual prices, expressed in the same unit as the original data (USD).

The evaluation results show an **RMSE value of 44,018**. This means that, on average, the ARIMA model’s Bitcoin price predictions deviate by approximately **44 thousand USD** from the actual prices during the testing period.

This relatively large RMSE should be interpreted within the context of Bitcoin’s characteristics. Bitcoin has a high price level and extreme volatility, so prediction errors in the tens of thousands of dollars are still reasonable for a simple time-series model such as ARIMA.

These results indicate that ARIMA is capable of capturing the **general direction of Bitcoin price movements**, but it still has limitations in tracking sharp price fluctuations. Therefore, ARIMA serves as a **baseline model**, and its performance will be compared with more complex models, such as SARIMA and Prophet, to assess whether alternative approaches can produce more accurate forecasts.


### 3.3 Model 2: SARIMA

In [ ]:
# Using order (1,1,1) and seasonal (1,1,1,12)
sarima_model = SARIMAX(train_data['Price'], 
                       order=(1, 1, 1), 
                       seasonal_order=(1, 1, 1, 12))
sarima_fit = sarima_model.fit(disp=False)

# Forecasting SARIMA
sarima_forecast = sarima_fit.forecast(steps=len(test_data))
sarima_rmse = np.sqrt(mean_squared_error(test_data['Price'], sarima_forecast))
print(f"SARIMA RMSE: {sarima_rmse:.2f}")

SARIMA RMSE: 36057.04


At this stage, the **SARIMA** model is used as an extension of ARIMA by incorporating a **seasonal** component. This model is designed to capture recurring periodic patterns that cannot be handled by standard ARIMA.

In the configuration **SARIMA(1,1,1)(1,1,1,12)**:

* The **(1,1,1)** component represents the non-seasonal pattern, similar to the ARIMA model
* The **(1,1,1,12)** component represents the seasonal pattern with a **12-month (annual)** cycle

The addition of **12** indicates that the model attempts to learn patterns that repeat every year. This is relevant for monthly Bitcoin price data, as price movements are often influenced by annual cycles such as market sentiment, macroeconomic conditions, and recurring investor behavior.

Similar to ARIMA, the **differencing** process is automatically applied within the model:

* **d = 1** removes long-term trends
* **D = 1** removes annual seasonal patterns

As a result, SARIMA operates on data that has been stabilized in terms of both trend and seasonality.

Evaluation results show that the **SARIMA RMSE is 36,057**, which is lower than the ARIMA RMSE. This reduction indicates that by accounting for seasonality, the model is able to produce more accurate predictions.

These findings suggest that although Bitcoin is known for its high volatility, there are still annual seasonal patterns that can be captured by the model. Therefore, SARIMA becomes a stronger alternative to ARIMA when the data exhibits recurring patterns over specific periods.

The SARIMA model is subsequently compared with the Prophet model to determine whether a more flexible approach to trends and pattern changes can achieve better predictive performance.

### 3.4 Model 3: Prophet

In [ ]:
# Prophet expects columns ['ds', 'y']
prophet_train = train_data.reset_index().rename(columns={'date': 'ds', 'Price': 'y'})
prophet_test = test_data.reset_index().rename(columns={'date': 'ds', 'Price': 'y'})

# Bitcoin often follows non-linear trends, 'multiplicative' seasonality may be more appropriate
m = Prophet(seasonality_mode='multiplicative')
m.fit(prophet_train)

future = m.make_future_dataframe(periods=len(test_data), freq='MS')
forecast = m.predict(future)

prophet_forecast = forecast.iloc[-len(test_data):]['yhat']
prophet_rmse = np.sqrt(mean_squared_error(test_data['Price'], prophet_forecast))
print(f"Prophet RMSE: {prophet_rmse:.2f}")

13:40:26 - cmdstanpy - INFO - Chain [1] start processing
13:40:26 - cmdstanpy - INFO - Chain [1] done processing


Prophet RMSE: 47777.32


At this stage, the **Prophet** model is applied, a time-series forecasting model developed by Facebook (Meta). This model is designed to simplify time-based forecasting, particularly for data that exhibits **non-linear trends**, **sudden pattern changes (changepoints)**, and potentially **irregular or messy observations**.

Unlike ARIMA and SARIMA, Prophet does not require the data to be stationary and does not explicitly rely on a *differencing* process. Instead, the model automatically decomposes the data into several key components:

* **Long-term trend**, which can evolve over time
* **Seasonality patterns**, such as yearly or monthly cycles
* **Special event effects**, if defined (e.g., holidays or specific events)

In this implementation, Prophet uses **multiplicative seasonality**, meaning seasonal effects are assumed to scale with the price level. This approach is relevant for Bitcoin, as price fluctuations tend to become larger when the price itself is at higher levels.

After training the model on the training dataset, Prophet is used to forecast Bitcoin prices over the test period. The evaluation results show an **RMSE of 47,777**, which is higher than both ARIMA and SARIMA.

These results indicate that although Prophet is highly flexible and powerful in handling trend changes and *changepoints*, its performance in this case is less optimal for predicting monthly Bitcoin prices. One possible reason is Bitcoin’s extreme volatility and sharp price movements, which may be difficult for smoother trend-based approaches like Prophet to capture accurately.

Therefore, in this project, Prophet serves as a comparison with a modern modeling approach, while classical models such as SARIMA demonstrate better performance in capturing monthly Bitcoin price patterns.

## 4. Deployment & Evaluation

### 4.1 Model Evaluation

In [43]:
results = pd.DataFrame({
    'Model': ['ARIMA', 'SARIMA', 'Prophet'],
    'RMSE': [arima_rmse, sarima_rmse, prophet_rmse]
})

results.sort_values(by='RMSE', ascending=True)

,Model,RMSE
1,SARIMA,36057.038513
0,ARIMA,44018.147801
2,Prophet,47777.318698


At this stage, a **model performance comparison** is conducted using the **RMSE (Root Mean Squared Error)** metric. The purpose of this step is to determine which model provides the most accurate predictions on the test dataset.

The comparison results indicate the following performance ranking:

1. **SARIMA** – RMSE ≈ **36,071**
2. **ARIMA** – RMSE ≈ **44,035**
3. **Prophet** – RMSE ≈ **47,793**

The model with the lowest RMSE is considered to have the best performance, as it produces smaller average differences between predicted and actual values. Based on these results, **SARIMA** emerges as the best-performing model for predicting monthly Bitcoin prices during the testing period.

SARIMA’s advantage likely comes from its ability to capture **annual seasonal patterns (12 months)**, which are not explicitly modeled in ARIMA and are handled more flexibly—but less precisely—by Prophet. This suggests that despite Bitcoin’s high volatility, there are still relatively consistent seasonal patterns in the monthly data.

ARIMA, as the *baseline* model, delivers reasonably good performance but falls behind SARIMA because it does not incorporate seasonal components. Meanwhile, Prophet—although effective in modeling non-linear trends and *changepoints*—shows the lowest performance in this case, possibly due to Bitcoin’s extreme volatility, which may be difficult for smoother trend-based approaches to capture accurately.

Based on this evaluation, **SARIMA is selected as the best model** to be used in the final visualization and business interpretation stage, particularly for analyzing medium-term Bitcoin price trend projections.


### 4.2 Final Forecast Visualization

In [ ]:
# Determine the best model automatically
best_model_name = results.sort_values(by='RMSE').iloc[0]['Model']

if best_model_name == 'ARIMA':
    best_forecast = arima_forecast
elif best_model_name == 'SARIMA':
    best_forecast = sarima_forecast
else:
    best_forecast = prophet_forecast

# Plotting Comparison
fig = go.Figure()

# Actual Data
fig.add_trace(go.Scatter(x=test_data.index, y=test_data['Price'], 
                         mode='lines', name='Actual Price'))

# Best Model Forecast
fig.add_trace(go.Scatter(x=test_data.index, y=best_forecast, 
                         mode='lines', name=f'{best_model_name} Forecast', 
                         line=dict(color='red', dash='dot')))

fig.update_layout(title=f'Bitcoin Price Prediction: Actual vs {best_model_name}',
                  xaxis_title='Date',
                  yaxis_title='Price (USD)')
fig.show()

At this stage, a **visual comparison between actual Bitcoin prices and predictions from the best-performing model**, **SARIMA**, is presented. The SARIMA model was automatically selected based on having the **lowest RMSE** compared to ARIMA and Prophet.

The chart displays:

* **Blue line (Actual Price)**: the actual Bitcoin prices during the test period (January 2024 – December 2025)
* **Red dashed line (SARIMA Forecast)**: the SARIMA model’s predicted Bitcoin prices for the same period

The main purpose of this visualization is to assess **how closely the model’s predictions align with actual price movements**, not only in terms of evaluation metrics (RMSE) but also regarding pattern and trend direction.

From the chart, it can be observed that:

* The SARIMA model performs reasonably well in capturing the **overall price movement direction** (upward or downward)
* However, SARIMA predictions appear **smoother** than the actual price series
* Sharp price spikes in the actual data are not fully captured by the model

This behavior is expected because SARIMA relies on historical and seasonal patterns, making it more focused on **average trends** rather than extreme short-term fluctuations. In other words, the model is more suitable for:

* Identifying **medium-term trend direction**
* Providing a general overview of price movements
* Supporting strategic analysis rather than precise short-term trading predictions

This visualization reinforces the previous evaluation results that **SARIMA is the best-performing model in this project**, while still acknowledging its limitations in capturing Bitcoin’s extreme volatility. Therefore, the forecast results should be interpreted as a **trend indicator** rather than exact price targets for short-term trading decisions.

### 4.3 Save to Joblib

In [45]:
# =========================
# Model configuration
# =========================
MODEL_NAME = "sarima"
MODEL_DISPLAY_NAME = "SARIMA"
MODEL_ORDER = "(1,1,1)(1,1,1,12)"

# Create Models directory (one level above notebook)
MODEL_DIR = "..\\models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

MODEL_FILENAME = f"{MODEL_NAME}_model_{timestamp}.joblib"
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_FILENAME)

joblib.dump(sarima_fit, MODEL_PATH)

print(f"{MODEL_DISPLAY_NAME} model successfully saved at: {MODEL_PATH}")

metadata = {
    "model_name": MODEL_DISPLAY_NAME,
    "model_key": MODEL_NAME,
    "model_order": MODEL_ORDER,
    "model_file": MODEL_FILENAME,
    "trained_at": timestamp,
    "train_period": {
        "start": str(train_data.index.min().date()),
        "end": str(train_data.index.max().date())
    },
    "test_period": {
        "start": str(test_data.index.min().date()),
        "end": str(test_data.index.max().date())
    },
    "evaluation_metric": "RMSE",
    "rmse": float(sarima_rmse),
    "data_frequency": "Monthly",
    "target_variable": "Bitcoin Close Price (USD)"
}

METADATA_FILENAME = f"{MODEL_NAME}_metadata_{timestamp}.json"
METADATA_PATH = os.path.join(MODEL_DIR, METADATA_FILENAME)

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Model metadata saved at: {METADATA_PATH}")

SARIMA model successfully saved at: ..\models\sarima_model_20251229_134026.joblib
Model metadata saved at: ..\models\sarima_metadata_20251229_134026.json


## 5. Stakeholder Acceptance

### 5.1 Conclusions

This project demonstrates that a *time-series forecasting* approach applied to **monthly Bitcoin data** provides a clearer view of price trends compared to highly volatile daily data. Among the three evaluated models—**ARIMA, SARIMA, and Prophet**—the **SARIMA** model achieved the best performance based on the **lowest RMSE**.

SARIMA’s advantage lies in its ability to capture **annual seasonal patterns (12 months)** that remain observable in Bitcoin price movements. The visualization results also show that SARIMA is reasonably effective in following the **general direction of price movements**, making it suitable as a baseline model for medium-term trend analysis.

---

### 5.2 Limitations

Despite producing promising results, this project has several limitations:

* The model relies solely on **historical price data**, without incorporating external factors such as news, regulations, macroeconomic conditions, or market sentiment.
* The forecast results tend to be **smoother** and are not able to fully capture extreme price spikes frequently observed in Bitcoin.
* The analysis is conducted at a **monthly scale**, making it less suitable for short-term or daily trading purposes.
* ARIMA and SARIMA parameters have not been fully optimized through systematic processes such as *grid search* or AutoARIMA.

---

### 5.3 Recommendations

Based on these findings and limitations, several recommendations can be considered:

* Use forecast results as a **trend direction indicator** rather than exact price targets.
* Combine time-series models with **external variables** such as trading volume, macroeconomic indicators, or market sentiment.
* Perform **model parameter optimization** to improve prediction accuracy.
* Develop a **hybrid approach**, for example by combining SARIMA with machine learning models.
* Adjust the prediction horizon according to business needs, particularly for medium- to long-term strategic analysis.
